# Suite DC — Dataclasses adapter

`mbse.Schemas.Adapters.Dataclasses` translates between Python dataclasses and object schemas, through `ast` trees in
both directions and never through source text. Fields of native types are properties; containers of dataclasses are
relations, seen through adjacencies. Defaults, mandatoriness and nesting are handled separately. The adapter is
Python's own, so this suite has no TypeScript counterpart. The trees "from other tools" below are built with `ast`
node constructors, not parsed.

In [ ]:
import ast
import dataclasses
import typing
from dataclasses import InitVar, dataclass

from mbse.Schemas.Adapters.Dataclasses import FromDataclass, ToDataclass
from mbse.Schemas.Framework import JSON, Proxies, Schemas as S, Validators
from support import raises, text

store = Proxies.OfStore()


def natives(schema):
    """The native type of each property, in order."""
    return [(name, prop.type.type) for name, prop in schema.properties.items()]


def views(schema):
    """Each adjacency's relation links and the link it is seen through."""
    return {name: (a.relation.links, a.me) for name, a in schema.adjacencies.items()}


def name(id_):
    return ast.Name(id=id_, ctx=ast.Load())


def subscript(outer, *items):
    return ast.Subscript(value=name(outer), slice=items[0] if len(items) == 1 else ast.Tuple(elts=list(items),
                         ctx=ast.Load()), ctx=ast.Load())


def field(id_, annotation):
    return ast.AnnAssign(target=ast.Name(id=id_, ctx=ast.Store()), annotation=annotation, simple=1)


def classdef(body, decorators=(name("dataclass"),), bases=(), called="Made"):
    """A class definition, as another tool might build it."""
    return ast.ClassDef(name=called, bases=list(bases), keywords=[], body=list(body), decorator_list=list(decorators),
                        type_params=[])

## DC-01 · A dataclass's native fields become properties, in order

In [ ]:
@dataclass
class Base:
    id: int


@dataclass
class Reading(Base):
    label: str
    value: float = 0.0  # defaults are not translated
    valid: "bool" = True  # string annotations are resolved
    raw: bytes = b""
    unit: typing.ClassVar[str] = "V"  # not a field
    scale: InitVar[int] = 1  # not a field


schema = FromDataclass(Reading)
assert natives(schema) == [("id", int), ("label", str), ("value", float), ("valid", bool), ("raw", bytes)]
assert schema.validate() == [] and schema.adjacencies == {}
assert list(FromDataclass.model(Reading)) == ["Reading"]


@dataclass
class Empty:
    pass


assert FromDataclass(Empty).properties == {}


@dataclass
class Series:
    tags: list[str]
    rows: typing.List[list[int]]
    deep: list[list[list[bytes]]]
    attrs: dict[str, list[float]]


series = FromDataclass(Series)  # lists of natives, at any depth, are lists
assert isinstance(series.properties["tags"].type, S.OfIndexed.Data) and series.properties["tags"].type.item.type is str
assert series.properties["rows"].type.item.item.type is int and series.properties["deep"].type.item.item.item.type is bytes
attrs = series.properties["attrs"].type  # a dict of natives by a native key other than int is a keyed list
assert not attrs.positional and attrs.key.type is str and attrs.item.item.type is float
assert series.adjacencies == {} and series.validate() == []

## DC-02 · FromDataclass refuses what is neither a native field nor a container of dataclasses

In [ ]:
class Plain:
    x: int


with raises(TypeError, match="Plain is not a dataclass"):
    FromDataclass(Plain)
with raises(TypeError, match="expected a dataclass or an ast.ClassDef, got int"):
    FromDataclass(3)


@dataclass
class Shapes:
    base: Base
    tags: set[str]
    pairs: dict[int, str]
    keyed: dict[Base, Base]
    bare: list
    tuples: tuple[Base, ...]
    either: int | str
    maybe: typing.Optional[int]
    anything: typing.Any
    choice: typing.Literal["a", "b"]
    nothing: None


containers = "a container field holds dataclasses, which become related objects"
problems = {
    "base": "field 'base': Base is a nested dataclass; nesting is handled separately",
    "tags": f"field 'tags': set holds str that is not a dataclass being translated; {containers}",
    "pairs": f"field 'pairs': dict holds str that is not a dataclass being translated; {containers}",
    "keyed": "field 'keyed': dict needs a native key type and a dataclass value type",
    "bare": "field 'bare': list needs a dataclass element type",
    "tuples": "field 'tuples': tuple is not translated; container fields are list, set or dict of dataclasses",
    "either": "field 'either': a union needs a name per branch; unions are not translated",
    "maybe": "field 'maybe': a union needs a name per branch; unions are not translated",
    "anything": "field 'anything': Any is neither a native type nor a dataclass being translated",
    "choice": "field 'choice': Literal is not translated",
    "nothing": "field 'nothing': the annotation is not a type",
}
base = FromDataclass.ast(Base)
for statement in FromDataclass.ast(Shapes).body:  # one field at a time, with Base in the model
    with raises(TypeError, match=problems[statement.target.id]):
        FromDataclass.model(classdef([statement]), base)


def another_base():
    @dataclass
    class Base:
        other: str

    return Base


with raises(ValueError, match="two classes are named 'Base'"):
    FromDataclass.model(Base, another_base())
with raises(ValueError, match="two classes are named 'Base'"):
    FromDataclass.model(base, Base)


@dataclass
class Twice:
    items: set[Base]
    Items: set[Base]  # the relation would be TwiceItems twice


with raises(ValueError, match="field 'Items': the relation name 'TwiceItems' is already used"):
    FromDataclass(Twice)


owner = classdef([field("things", subscript("set", name("Clash")))], called="Owner")  # gives Clash 'owner_things'
with raises(ValueError, match="Clash: ['owner_things'] would be both properties and adjacencies"):
    FromDataclass.model(owner, classdef([field("owner_things", name("int"))], called="Clash"))
with raises(ValueError, match="A: the adjacency 'b_things' would be declared twice"):  # A's field, and B's inverse view
    FromDataclass.model(classdef([field("b_things", subscript("set", name("A")))], called="A"),
                        classdef([field("things", subscript("set", name("A")))], called="B"))

## DC-03 · Containers of dataclasses become relations, seen through adjacencies

In [ ]:
@dataclass
class Address:
    street: str


@dataclass
class Place:
    title: str


@dataclass
class Person:
    name: str
    homes: list[Address | Place]
    labels: dict[str, Address]
    children: set["Person"]


model = FromDataclass.model(Person)
assert list(model) == ["Person", "Address", "Place", "PersonHomes", "PersonLabels", "PersonChildren"]
assert all(schema.validate() == [] for schema in model.values())
person, homes, labels, children = model["Person"], model["PersonHomes"], model["PersonLabels"], model["PersonChildren"]
assert natives(person) == [("name", str)]
assert views(person) == {"homes": (("owner", "item"), "owner"), "labels": (("owner", "item"), "owner"),
                         "children": (("owner", "item"), "owner"), "person_children": (("owner", "item"), "item")}
assert views(model["Address"]) == {"person_homes": (("owner", "item"), "item"),
                                   "person_labels": (("owner", "item"), "item")}
assert views(model["Place"]) == {"person_homes": (("owner", "item"), "item")}  # a union: both may fill 'item'
assert person.adjacencies["homes"].relation is homes and model["Place"].adjacencies["person_homes"].relation is homes
assert natives(homes) == [("index", int)] and homes.uniques == (frozenset({"item"}),)  # owner and index give the item
assert natives(labels) == [("key", str)] and labels.uniques == (frozenset({"item"}),)
assert natives(children) == [] and children.uniques == ()
assert FromDataclass(Person) is not person and natives(FromDataclass(Person)) == [("name", str)]

# The schemas work like any others.
assert all(schema.name == name for name, schema in model.items())  # named after the classes and fields
for schema in model.values():
    store.register(schema)
B = store
home = B.Address().street("1 Main").create()
ann = B.Person().name("Ann").homes(lambda e: e.item(home).index(0)).labels(lambda e: e.item(home).key("home")).create()
B.Person(ann).children(lambda e: e.item(lambda p: p.name("Bo"))).update()
assert Validators.Validate(B).Reachable(person, ann) == []
assert JSON.ToJSON(store)(person, ann) == ('{"root": "s0", "objects": {"s0": {"name": "Ann", "homes": [{"item": {"$ref": "s1", '
                                    '"$schema": "Address"}, "index": 0}], "labels": [{"item": {"$ref": "s1", "$schema": '
                                    '"Address"}, "key": "home"}], "children": [{"item": {"$ref": "s2", "$schema": '
                                    '"Person"}}]}}}')

## DC-04 · FromDataclass reads trees, from FromDataclass.ast or built by other tools

In [ ]:
tree = FromDataclass.ast(Reading)
assert isinstance(tree, ast.ClassDef) and tree.name == "Reading" and tree.bases == []
assert [s.target.id for s in tree.body] == ["id", "label", "value", "valid", "raw"]
assert [s.annotation.id for s in tree.body] == ["int", "str", "float", "bool", "bytes"]
assert [s.value for s in tree.body] == [None] * 5  # no defaults
assert [d.id for d in tree.decorator_list] == ["dataclass"]
assert isinstance(FromDataclass.ast(Empty).body[0], ast.Pass)
homes_annotation = FromDataclass.ast(Person).body[1].annotation
assert homes_annotation.value.id == "list" and isinstance(homes_annotation.slice, ast.BinOp)
labels_annotation = FromDataclass.ast(Person).body[2].annotation
assert labels_annotation.value.id == "dict" and [e.id for e in labels_annotation.slice.elts] == ["str", "Address"]

typing_ = name("typing")
made = classdef([
    ast.Expr(value=ast.Constant(value="A docstring")),
    field("count", name("int")),
    field("note", ast.Attribute(value=name("builtins"), attr="str", ctx=ast.Load())),
    field("limit", ast.Subscript(value=ast.Attribute(value=typing_, attr="ClassVar", ctx=ast.Load()), slice=name("int"),
                                 ctx=ast.Load())),
    field("seed", name("InitVar")),
    field("friends", ast.Subscript(value=ast.Attribute(value=typing_, attr="FrozenSet", ctx=ast.Load()),
                                   slice=name("Made"), ctx=ast.Load())),
    ast.AnnAssign(target=ast.Attribute(value=name("self"), attr="x", ctx=ast.Store()), annotation=name("int"), simple=0),
    ast.Assign(targets=[ast.Name(id="plain", ctx=ast.Store())], value=ast.Constant(value=1)),
    ast.FunctionDef(name="method", args=ast.arguments(posonlyargs=[], args=[], kwonlyargs=[], kw_defaults=[],
                    defaults=[]), body=[ast.Pass()], decorator_list=[], type_params=[]),
], decorators=[ast.Call(func=ast.Attribute(value=name("dataclasses"), attr="dataclass", ctx=ast.Load()), args=[],
                        keywords=[ast.keyword(arg="frozen", value=ast.Constant(value=True))])])
made_model = FromDataclass.model(made)
assert natives(made_model["Made"]) == [("count", int), ("note", str)]
assert views(made_model["Made"]) == {"friends": (("owner", "item"), "owner"), "made_friends": (("owner", "item"), "item")}
assert natives(FromDataclass(made)) == [("count", int), ("note", str)]
with raises(TypeError, match="Made is not decorated with @dataclass"):
    FromDataclass(classdef([field("count", name("int"))], decorators=[name("other")]))
with raises(TypeError, match="Made has base classes; pass the class itself, so its inherited fields are included"):
    FromDataclass(classdef([field("count", name("int"))], bases=[name("Base")]))
with raises(TypeError, match="field 'x': a union needs a name per branch"):
    FromDataclass(classdef([field("x", subscript("Optional", name("int")))]))
with raises(TypeError, match="field 'x': this generic type is not translated"):
    FromDataclass(classdef([field("x", ast.Subscript(value=ast.Constant(value=1), slice=name("int"), ctx=ast.Load()))]))
with raises(TypeError, match="field 'x': dict needs a native key type and a dataclass value type"):
    FromDataclass(classdef([field("x", subscript("dict", name("Made")))]))
with raises(TypeError, match="field 'x': set holds something that is not a dataclass being translated"):
    FromDataclass(classdef([field("x", subscript("set", ast.Constant(value=None)))]))

## DC-05 · ToDataclass writes a dataclass with one field per property

In [ ]:
Id = S.OfUnion.Builder().branches(lambda b: b.name("number").of(lambda t: t.as_native(int)),
                                  lambda b: b.name("code").of(lambda t: t.as_native(str))).create()
Sensor = S.OfObject.Builder().properties(text("name"), text("rate", float), text("count", int)).create()
Made = ToDataclass(Sensor, "Sensor")
assert ToDataclass(S.OfObject.Builder(Sensor).name("iot.Sensor").clone()).__name__ == "Sensor"  # after the last part
with raises(ValueError, match="an unnamed schema needs a class name: ToDataclass(schema, name)"):
    ToDataclass(Sensor)
assert dataclasses.is_dataclass(Made) and Made.__name__ == "Sensor"
assert Made.__module__ == "mbse.Schemas.Adapters.Dataclasses"
assert typing.get_type_hints(Made) == {"name": str, "rate": float, "count": int}
assert [f.name for f in dataclasses.fields(Made)] == ["name", "rate", "count"]
assert all(f.default is dataclasses.MISSING for f in dataclasses.fields(Made))  # no defaults
assert Made("s1", 2.5, 3).rate == 2.5
assert dataclasses.is_dataclass(ToDataclass(S.OfObject.Builder().create(), "Nothing"))
Lists = S.OfObject.Builder().properties(
    lambda p: p.name("tags").of(lambda t: t.as_indexed(lambda i: i.of(lambda t: t.as_native(str)))),
    lambda p: p.name("rows").of(lambda t: t.as_indexed(lambda i: i.of(lambda t: t.as_indexed(
        lambda j: j.of(lambda t: t.as_native(int))))))).create()
assert typing.get_type_hints(ToDataclass(Lists, "Lists")) == {"tags": list[str], "rows": list[list[int]]}
Keyed = S.OfObject.Builder().properties(lambda p: p.name("attrs").of(lambda t: t.as_indexed(
    lambda i: i.key(lambda t: t.as_native(str)).of(lambda t: t.as_native(float))))).create()
assert typing.get_type_hints(ToDataclass(Keyed, "Keyed")) == {"attrs": dict[str, float]}
assert FromDataclass(ToDataclass(Keyed, "Keyed")).properties["attrs"].type.key.type is str
again = FromDataclass(ToDataclass(Lists, "Lists"))
assert again.properties["tags"].type.item.type is str and again.properties["rows"].type.item.item.type is int

tree = ToDataclass.ast({"Sensor": Sensor})
assert isinstance(tree, ast.Module) and [type(s) for s in tree.body] == [ast.ClassDef]
assert [s.target.id for s in tree.body[0].body] == ["name", "rate", "count"]
assert list(ToDataclass.compile(tree)) == ["Sensor"]
Plainer = S.OfObject.Builder().properties(text("name"), text("rate", float)).create()
assert natives(FromDataclass(ToDataclass(Plainer, "Plainer"))) == natives(Plainer)
assert [(f.name, f.type) for f in dataclasses.fields(ToDataclass(FromDataclass(Reading), "Reading"))] == [
    ("id", "int"), ("label", "str"), ("value", "float"), ("valid", "bool"), ("raw", "bytes")]  # postponed annotations

## DC-06 · ToDataclass writes adjacencies as containers, typed by the schemas that may fill the other link

In [ ]:
classes = ToDataclass.model(model)  # relation schemas are skipped
assert list(classes) == ["Person", "Address", "Place"]
hints = {name_: typing.get_type_hints(cls, localns=classes) for name_, cls in classes.items()}
assert hints["Person"] == {"name": str, "homes": list[classes["Address"] | classes["Place"]],
                           "labels": dict[str, classes["Address"]], "children": set[classes["Person"]]}
assert hints["Address"] == {"street": str} and hints["Place"] == {"title": str}  # 'item' views are not fields

# Round trips.
again = FromDataclass.model(*classes.values())
assert list(again) == list(model)
assert all(natives(again[n]) == natives(model[n]) and views(again[n]) == views(model[n]) for n in classes)
assert all((again[n].links, natives(again[n]), again[n].uniques) == (model[n].links, natives(model[n]), model[n].uniques)
           for n in ["PersonHomes", "PersonLabels", "PersonChildren"])

# Any two-link relation: the first link is the container's owner.
Card = S.OfObject.Builder().properties(text("name")).create()
Street = S.OfObject.Builder().properties(text("line")).create()
Lives = S.OfRelation.Builder().links("card", "street").create()
Owns = S.OfRelation.Builder().links("parent", "child").unique("parent").create()  # cardinality does not matter
S.OfObject.Builder(Card).relations(lambda r: r.name("streets").of(Lives).me("card"),
                                   lambda r: r.name("kids").of(Owns).me("parent"),
                                   lambda r: r.name("parent").of(Owns).me("child")).update()
S.OfObject.Builder(Street).relations(lambda r: r.name("cards").of(Lives).me("street")).update()
cards = ToDataclass.model({"Card": Card, "Street": Street})
assert typing.get_type_hints(cards["Card"], localns=cards) == {"name": str, "streets": set[cards["Street"]],
                                                               "kids": set[cards["Card"]]}
assert list(typing.get_type_hints(cards["Street"], localns=cards)) == ["line"]

Hub = S.OfObject.Builder().properties(text("name")).create()
Three = S.OfRelation.Builder().links("a", "b", "c").create()
Tagged = S.OfRelation.Builder().links("a", "b").properties(text("tag")).create()
Keyed = S.OfRelation.Builder().links("a", "b").properties(lambda p: p.name("at").of(Id)).unique("b").create()
Lonely = S.OfRelation.Builder().links("a", "b").create()
cases = [
    (Three, "adjacency 'x': only a relation with two links has a container form"),
    (Tagged, "adjacency 'x': only one native property and unique('b') have a container form"),
    (Keyed, "adjacency 'x': only one native property and unique('b') have a container form"),
    (Lonely, "adjacency 'x': no object schema declares an adjacency via 'b', so the element type is unknown"),
]
for relation, message in cases:
    Holder = S.OfObject.Builder().relations(lambda r, rel=relation: r.name("x").of(rel).me("a")).create()
    if relation is not Lonely:
        S.OfObject.Builder(Holder).relations(lambda r, rel=relation: r.name("y").of(rel).me("b")).update()
    with raises(TypeError, match=message):
        ToDataclass(Holder, "Holder")

## DC-07 · ToDataclass refuses what a flat dataclass cannot hold

In [ ]:
Point = S.OfObject.Builder().properties(text("x", int)).create()
Place_ = S.OfObject.Builder().properties(lambda p: p.name("at").of(Point)).create()
with raises(TypeError, match="property 'at' holds a value object; nested dataclasses are handled separately"):
    ToDataclass(Place_, "Spot")
with raises(TypeError, match="property 'id' holds a union value; nested dataclasses are handled separately"):
    ToDataclass(S.OfObject.Builder().properties(lambda p: p.name("id").of(Id)).create(), "Tag")
Points = S.OfObject.Builder().properties(lambda p: p.name("points").of(lambda t: t.as_indexed(lambda i: i.of(Point)))).create()
with raises(TypeError, match="property 'points' holds a value object; nested dataclasses are handled separately"):
    ToDataclass(Points, "Points")
Count = S.OfIntersection.Builder().parts(lambda p: p.name("low").of(lambda t: t.as_native(int)),
                                         lambda p: p.name("high").of(lambda t: t.as_native(int))).create()
with raises(TypeError, match="property 'c' holds an intersection value; nested dataclasses are handled separately"):
    ToDataclass(S.OfObject.Builder().properties(lambda p: p.name("c").of(Count)).create(), "C")
with raises(TypeError, match="expected an object schema, got _NativeData"):
    ToDataclass(S.OfNative.Data(int), "N")
with raises(TypeError, match="'N': expected an object or relation schema, got _NativeData"):
    ToDataclass.model({"N": S.OfNative.Data(int)})
with raises(ValueError, match="class name '1st' is not a Python identifier"):
    ToDataclass(Point, "1st")
for reserved in ["dataclass", "int", "set"]:
    with raises(ValueError, match=f"class name '{reserved}' is reserved"):
        ToDataclass(Point, reserved)
with raises(ValueError, match="property 'class' is not a Python identifier"):
    ToDataclass(S.OfObject.Builder().properties(text("class")).create(), "Lesson")
Self_ = S.OfRelation.Builder().links("a", "b").create()
Odd = S.OfObject.Builder().relations(lambda r: r.name("for").of(Self_).me("a"), lambda r: r.name("y").of(Self_).me("b")).create()
with raises(ValueError, match="adjacency 'for' is not a Python identifier"):
    ToDataclass(Odd, "Odd")
for bad in [ast.Expression(body=name("x")), ast.Module(body=[], type_ignores=[]),
            ast.Module(body=[ast.Pass()], type_ignores=[])]:
    with raises(ValueError, match="expected an ast.Module of class definitions"):
        ToDataclass.compile(bad)